In [0]:
%sql
create or replace table workspace.default.study_compute_events_s03_bronze as
select * 
from values
    (1, '2026-08-23', 'SUCCESS', 120),
    (2, '2026-08-23', 'FAILED', 80),
    (3, '2026-08-23', 'SUCCESS', 150),
    (4, '2026-08-24', 'INVALID', 90),
    (5, '2026-08-24', 'SUCCESS', 110),
    (5, '2026-08-24', 'SUCCESS', 110)
as events(event_id, event_date, status, amount);

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- construisons la silver

create or replace table workspace.default.study_events_s03_silver as
select distinct
    event_id,
    cast(event_date as date) as event_date,
    status,
    amount
from workspace.default.study_compute_events_s03_bronze
where status in ('SUCCESS', 'FAILED');

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.default.study_events_s03_silver order by event_id

event_id,event_date,status,amount
1,2026-08-23,SUCCESS,120
2,2026-08-23,FAILED,80
3,2026-08-23,SUCCESS,150
5,2026-08-24,SUCCESS,110


In [0]:
%sql
-- construire la couche gold

create or replace table workspace.default.study_compute_events_s03_bronze

select 
    event_date,
    status,
    count(*) as event_count,
    sum(amount) as total_amount

from workspace.default.study_events_s03_silver
group by 
    event_date,
    status

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.default.study_compute_events_s03_bronze order by event_date, status

event_date,status,event_count,total_amount
2026-08-23,FAILED,1,80
2026-08-23,SUCCESS,2,270
2026-08-24,SUCCESS,1,110
